# Naive RAG Pipeline — Relationship Dataset Wikipedia (Cohere)

This notebook implements a complete **Naive RAG** pipeline powered entirely by **Cohere**:

| Step | Cohere component |
|------|------------------|
| **Retrieval** | `embed-english-v3.0` — dense embeddings + cosine similarity |
| **Generation** | `command-r-plus` — grounded answer generation via Documents API |

### Pipeline Overview
```
JSON Chunks → Cohere Embeddings → Cosine Search → command-r-plus Answer → LLM as a judge
```

### Evaluation Metrics
We will utilise LLM as a judge and attach 3 criterias to it as follows
- faithfulness:      Every claim in the answer is directly supported by the provided context.
                     Penalise heavily for hallucination or any statement not grounded in the context.
- answer_relevance:  The answer directly and completely addresses the question with a real answer.
                     IMPORTANT: If the answer says it "cannot answer", "no information", "not found in
                     the graph", or anything similar — that is a FAILED answer. Score it 1.
- context_relevance: The retrieved context actually contained information useful for answering
                     the question. If the context is off-topic or the answer says no info was found,
                     score this 1.

<h1>Preprocessing for Relationship Dataset - Wikipedia</h1>

In [1]:
import json
import re

# ── config ────────────────────────────────────────────────────────────────────

MIN_CHUNK_WORDS = 30   # discard paragraphs that are too short to be useful

# ── helpers ───────────────────────────────────────────────────────────────────

def make_id(title: str, idx: int) -> str:
    """e.g. 'Artificial Intelligence' -> 'artificial_intelligence_chunk_001'"""
    slug = re.sub(r'\s+', '_', title.strip().lower())
    slug = re.sub(r'[^a-z0-9_]', '', slug)
    return f"{slug}_chunk_{idx:03d}"

def split_paragraphs(text: str) -> list[str]:
    """Split on blank lines, clean whitespace."""
    paragraphs = re.split(r'\n{2,}', text)
    cleaned = []
    for p in paragraphs:
        p = p.strip()
        p = re.sub(r'\s+', ' ', p)       # collapse internal whitespace
        p = re.sub(r'\[\d+\]', '', p)    # strip citation markers like [12]
        if len(p.split()) >= MIN_CHUNK_WORDS:
            cleaned.append(p)
    return cleaned

# ── main converter ────────────────────────────────────────────────────────────

def raw_to_naive_rag(records: list[dict]) -> dict:
    chunks = []
    for record in records:
        title = record['title']
        text  = record['content']         # 'content' key (not 'text')
        paragraphs = split_paragraphs(text)
        for idx, para in enumerate(paragraphs):
            chunks.append({
                'id':   make_id(title, idx),
                'text': f"{title}: {para}",  # prepend title for retrieval context
            })
    seen = {}
    for c in chunks:
        seen[c['id']] = c
    deduped = list(seen.values())

    return {
        'total':  len(deduped),
        'chunks': deduped
    }

# ── entry point ───────────────────────────────────────────────────────────────

if __name__ == '__main__':
    with open('wiki_corpus.json', 'r', encoding='utf-8') as f:
        records = json.load(f)

    print(f"Loaded {len(records)} articles")

    result = raw_to_naive_rag(records)

    with open('wiki_corpus_naive_rag.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"Done — {result['total']} chunks written to wiki_corpus_naive_rag.json")

Loaded 100 articles
Done — 1732 chunks written to wiki_corpus_naive_rag.json


## 0 · Install dependencies

In [2]:
# !pip install -q cohere numpy tqdm

## 1 · Imports & Configuration

In [3]:
import json, re, string, os
from collections import Counter

import numpy as np
import cohere
from tqdm import tqdm

# ── Cohere configuration ──────────────────────────────────────────────────────
EMBED_MODEL    = "embed-english-v3.0"   # embedding model for retrieval
LLM_MODEL      = "command-a-03-2025"       # LLM for answer generation
TOP_K          = 5                      # number of chunks to retrieve per query
JSON_PATH      = "wiki_corpus_naive_rag.json"

ACTIVE_DATASET = "test_questions_wiki"

co = cohere.Client(api_key="bPxPN4MX7cZoTVNIK4zllf9tUGodwyYMTlKb64be")
print("✅ Cohere client ready")
print(f"   Embed model : {EMBED_MODEL}")
print(f"   LLM model   : {LLM_MODEL}")
print(f"   LLM model   : {JSON_PATH}")


✅ Cohere client ready
   Embed model : embed-english-v3.0
   LLM model   : command-a-03-2025
   LLM model   : wiki_corpus_naive_rag.json


## 2 · Load & Inspect the Knowledge Base

In [4]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Filter out table-of-contents stub chunks (text == "PAGE" or empty)
chunks = [c for c in raw["chunks"] if c["text"].strip() and c["text"].strip() != "PAGE"]

print(f"Total chunks in file : {raw['total']}")
print(f"Usable chunks        : {len(chunks)}")
print(f"\nSample chunk:")
print(json.dumps(chunks[10], indent=2))

Total chunks in file : 1732
Usable chunks        : 1732

Sample chunk:
{
  "id": "artificial_intelligence_chunk_010",
  "text": "Artificial intelligence: ==== State space search ==== State space search searches through a tree of possible states to try to find a goal state. For example, planning algorithms search through trees of goals and subgoals, attempting to find a path to a target goal, a process called means-ends analysis. Simple exhaustive searches are rarely sufficient for most real-world problems: the search space (the number of places to search) quickly grows to astronomical numbers. The result is a search that is too slow or never completes. \"Heuristics\" or \"rules of thumb\" can help prioritize choices that are more likely to reach a goal. Adversarial search is used for game-playing programs, such as chess or Go. It searches through a tree of possible moves and countermoves, looking for a winning position."
}


## 3 · Build the Dense Retriever with Cohere Embeddings

We embed every chunk once upfront and store the vectors in memory.
At query time we embed the question and compute cosine similarity against the corpus.


In [5]:
import time

def embed_texts(texts: list[str], input_type: str, batch_size: int = 48) -> np.ndarray:
    """
    Embed a list of texts in batches using Cohere embed-english-v3.0.

    input_type:
        'search_document' — for the knowledge-base corpus
        'search_query'    — for the user query at retrieval time
    """
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding ({input_type})"):
        batch = texts[i : i + batch_size]
        response = co.embed(
            texts=batch,
            model=EMBED_MODEL,
            input_type=input_type,
        )
        all_embeddings.extend(response.embeddings)
        time.sleep(10)
    return np.array(all_embeddings, dtype=np.float32)


# Use text so embeddings capture document structure
corpus_texts = [f"{c['text']}" for c in chunks]

# Embed the full corpus once (this may take ~1-2 min for ~400 chunks)
corpus_embeddings = embed_texts(corpus_texts, input_type="search_document")

print(f"\nCorpus embedding matrix : {corpus_embeddings.shape}")
print(f"Embedding dimension     : {corpus_embeddings.shape[1]}")


Embedding (search_document): 100%|██████████| 37/37 [06:44<00:00, 10.93s/it]


Corpus embedding matrix : (1732, 1024)
Embedding dimension     : 1024


In [6]:
def cosine_scores(query_vec: np.ndarray, corpus_vecs: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between one query vector and the corpus matrix."""
    q_norm = query_vec  / (np.linalg.norm(query_vec) + 1e-10)
    c_norm = corpus_vecs / (np.linalg.norm(corpus_vecs, axis=1, keepdims=True) + 1e-10)
    return c_norm @ q_norm


def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    """Embed query and return the top-k most relevant chunks."""
    resp      = co.embed(texts=[query], model=EMBED_MODEL, input_type="search_query")
    query_vec = np.array(resp.embeddings[0], dtype=np.float32)

    scores  = cosine_scores(query_vec, corpus_embeddings)
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        chunk = chunks[idx].copy()
        chunk["score"] = float(scores[idx])
        results.append(chunk)
    return results


# Sanity check
sample = retrieve("What is the relationship between artificial intelligence and machine learning?")
for r in sample:
    print(f"[{r['score']:.4f}] {r}")

[0.6443] {'id': 'artificial_intelligence_chunk_005', 'text': 'Artificial intelligence: === Learning === Machine learning is the study of programs that can improve their performance on a given task automatically. It has been a part of AI from the beginning.', 'score': 0.6443383693695068}
[0.6374] {'id': 'machine_learning_chunk_000', 'text': 'Machine learning: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions. Within a subdiscipline of machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance. Statistics and mathematical optimisation (mathematical programming) methods compose the foundations of machine learning. Data mining is a related field of 

## 4 · LLM Answer Generation with `command-r-plus`

We use Cohere's **Chat** endpoint with the `documents` parameter.
This passes retrieved chunks as structured grounding context, which reduces hallucination
and keeps answers faithful to the rulebook.

In [7]:
PREAMBLE = """\
You are a knowledgeable general knowledge assistant.
Answer the user's question using ONLY the provided document excerpts from Wikipedia articles.
Be concise and factual. If the answer is not in the documents, say "Not found in the provided context."
"""

def generate_answer(query: str, retrieved_chunks: list[dict]) -> str:
    """Call command-r-plus with retrieved chunks passed as grounded documents."""
    documents = [
        {
            "id"      : c["id"],
            "title"   : c["id"],
            "snippet" : c["text"],
        }
        for c in retrieved_chunks
    ]
    response = co.chat(
        model=LLM_MODEL,
        preamble=PREAMBLE,
        message=query,
        documents=documents,
        temperature=0,
        max_tokens=300,
    )
    return response.text.strip()

def rag_answer(query: str, top_k: int = TOP_K) -> tuple[str, list[dict]]:
    """Full RAG pipeline: retrieve → generate."""
    retrieved = retrieve(query, top_k=top_k)
    answer    = generate_answer(query, retrieved)
    return answer, retrieved

# Demo
q = "What is the relationship between artificial intelligence and machine learning?"
ans, ctx = rag_answer(q)
print(f"Q: {q}\nA: {ans}")

Q: What is the relationship between artificial intelligence and machine learning?
A: Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions.


## 5 · Define Evaluation Dataset

Each entry has a **question** and a short **reference answer** (ground truth).

> 💡 Add more QA pairs here to make the evaluation more robust.

In [8]:
with open(f"{ACTIVE_DATASET}.json", "r") as f:
    eval_dataset = json.load(f)
    
print(f"Evaluation dataset size: {len(eval_dataset)} questions")



Evaluation dataset size: 24 questions


## 6 · LLM as a judge

In [9]:
test_questions = eval_dataset

In [10]:
def chat_with_retry(max_retries=4, **kwargs):
    """Wraps co.chat with exponential backoff on TooManyRequestsError."""
    for attempt in range(max_retries):
        try:
            return co.chat(**kwargs)
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 15 * (2 ** attempt)  # 15s, 30s, 60s, 120s
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise  # re-raise non-rate-limit errors immediately
    raise RuntimeError("Max retries exceeded on Cohere rate limit.")

In [11]:
# ── Step 2: LLM Judge ──────────────────────────────────────────────────────
# Scores a single (question, context, answer) triple on 3 metrics.

def llm_judge(question, context, answer_text):
    """Score an answer using an LLM judge.

    Returns a dict with scores (1–5) for:
        faithfulness      – every claim is supported by the context
        answer_relevance  – answer directly addresses the question
        context_relevance – retrieved context was useful for the question

    A reasoning string is also returned for transparency.
    """
    prompt = f"""You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems.

Score the answer below on three criteria using a 1–5 integer scale:

  1 = very poor  |  3 = acceptable  |  5 = excellent

Criteria:
- faithfulness:      Every claim in the answer is directly supported by the provided context.
                     Penalise heavily for hallucination or any statement not grounded in the context.
- answer_relevance:  The answer directly and completely addresses the question with a real answer.
                     IMPORTANT: If the answer says it "cannot answer", "no information", "not found in
                     the graph", or anything similar — that is a FAILED answer. Score it 1.
- context_relevance: The retrieved context actually contained information useful for answering
                     the question. If the context is off-topic or the answer says no info was found,
                     score this 1.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context if context else "(no context retrieved)"}

ANSWER:
{answer_text}

Return ONLY a valid JSON object (no extra text):
{{
  "faithfulness": <1-5>,
  "answer_relevance": <1-5>,
  "context_relevance": <1-5>,
  "reasoning": "<one sentence explaining the scores>"
}}
"""
    response = chat_with_retry(model="command-r7b-12-2024", message=prompt, temperature=0)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        scores = json.loads(raw)
        # Clamp scores to valid range
        for key in ("faithfulness", "answer_relevance", "context_relevance"):
            scores[key] = max(1, min(5, int(scores.get(key, 1))))
        return scores
    except (json.JSONDecodeError, ValueError):
        return {"faithfulness": 0, "answer_relevance": 0, "context_relevance": 0,
                "reasoning": "parse error"}

# Quick smoke test
_test = llm_judge(
    "What is machine learning?",
    "Machine learning -[is part of]-> Artificial intelligence",
    "Machine learning is a subfield of artificial intelligence."
)
print("Judge smoke test:", _test)

Judge smoke test: {'faithfulness': 5, 'answer_relevance': 5, 'context_relevance': 5, 'reasoning': 'The answer is directly supported by the context, is relevant to the question, and provides a complete and accurate response.'}


In [12]:
# ── Step 3: Run evaluation ─────────────────────────────────────────────────
# Saves results after every question so a crash won't lose progress.
# Re-running this cell will skip already-completed questions automatically.

EVAL_RESULTS_PATH = f"data/eval_results_{ACTIVE_DATASET}.json"

# Resume from existing results if available
if os.path.exists(EVAL_RESULTS_PATH):
    with open(EVAL_RESULTS_PATH, "r", encoding="utf-8") as f:
        eval_results = json.load(f)
    done_questions = {r["question"] for r in eval_results}
    print(f"Resuming — {len(eval_results)} already done, {len(test_questions) - len(done_questions)} remaining.")
else:
    eval_results = []
    done_questions = set()

for i, item in enumerate(test_questions):
    question = item["question"]

    if question in done_questions:
        print(f"[{i+1}/{len(test_questions)}] Skipping (already done): {question[:60]}")
        continue

    print(f"[{i+1}/{len(test_questions)}] {question}")

    for attempt in range(3):
        try:
            result    = rag_answer(question)
            ans       = result[0]
            retrieved = result[1]                          # raw chunks
            ctx       = " | ".join(c["text"] for c in retrieved)  # string for llm_judge
            scores    = llm_judge(question, ctx, ans)
            break
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 20 * (attempt + 1)
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/3)...")
                time.sleep(wait)
            else:
                raise
    else:
        print(f"  Skipping after 3 failed attempts.")
        continue

    reference = item.get("reference_answer") or item.get("reference", "")

    eval_results.append({
        "question":          question,
        "reference_answer":  reference,
        "naiveRAG_answer":   ans,
        "context":           retrieved,
        "faithfulness":      scores["faithfulness"],
        "answer_relevance":  scores["answer_relevance"],
        "context_relevance": scores["context_relevance"],
        "reasoning":         scores.get("reasoning", ""),
    })

    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)

    time.sleep(4)

print(f"\nDone. {len(eval_results)}/{len(test_questions)} results saved to {EVAL_RESULTS_PATH}")

# ── Summary table ──────────────────────────────────────────────────────────
metrics = ("faithfulness", "answer_relevance", "context_relevance")
averages = {m: sum(r[m] for r in eval_results) / len(eval_results) for m in metrics}

print(f"\n{'='*45}")
print(f"  LLM as a judge Evaluation  —  dataset: {ACTIVE_DATASET}")
print(f"{'='*45}")
print(f"  Questions evaluated : {len(eval_results)}")
print(f"  Faithfulness        : {averages['faithfulness']:.2f} / 5")
print(f"  Answer Relevance    : {averages['answer_relevance']:.2f} / 5")
print(f"  Context Relevance   : {averages['context_relevance']:.2f} / 5")
avg_overall = sum(averages.values()) / len(averages)
print(f"  Overall             : {avg_overall:.2f} / 5")
print(f"{'='*45}")

# Per-question breakdown
print("\nPer-question scores (F=Faithfulness, A=Answer Rel., C=Context Rel.):")
print(f"{'#':<4} {'F':>3} {'A':>3} {'C':>3} Question")
print("-" * 70)
for i, r in enumerate(eval_results, 1):
    q_preview = r["question"][:48] + "…" if len(r["question"]) > 48 else r["question"]
    print(f"{i:<4} {r['faithfulness']:>3} {r['answer_relevance']:>3} {r['context_relevance']:>3} {q_preview}")

[1/24] How does the criss-cross algorithm differ from the simplex algorithm in terms of its approach to solving linear programming problems?
[2/24] What is the significance of the Klee–Minty cube in the context of the criss-cross algorithm and the simplex algorithm?
[3/24] How do the criss-cross algorithm and the simplex algorithm handle the Klee–Minty cube differently in terms of their average number of corner visits?
[4/24] How does the Bellman-Ford algorithm handle graphs with negative edge weights?
[5/24] What is the relationship between the Bellman-Ford algorithm and Dijkstra's algorithm?
[6/24] Who contributed to the development of the Bellman-Ford algorithm, and how is it named?
[7/24] How did the publication of 'Artificial Intelligence: A Modern Approach' influence the field of AI?
[8/24] What is the relationship between the topics covered in AIMA and the development of AI?
[9/24] How has the GitHub repository for AIMA contributed to the learning experience of AI students?
[10/